# The goal of this notebook is to extract player ids data from different tiers to compose the initial seed puuids that will help us collect the matches from that rank
## Sub Goals:
        1) Get 10 player gameName, tagLine, and puuids from each Rank [Iron, Bronze, Silver, Gold, Platnium, Emerald, Diamond, Masters, Grandmasters, Challenger] and division
        2) Combine them into a single CSV file for later usage

## Result:
        1) lower_rank_player_ids_collected = (7 Ranks) * (10 Players) * (4 Divisions) 
                == 280 player ids
        2) high_rank_player_ids_collected = (3 Ranks) * (10 Players) * (1 Division)
                == 30 player ids
        3) total_player_ids_collected = lower_rank_player_ids_collected + high_rank_player_ids_collected
                == 310 player ids

## Notes:
        1) Why 10 players per division? 
                - Just in case if 1 player does not have enough matches, we will use the next player in this rank + division to complete the missing matches.
                - Later on in the upcoming step, we will take 1 puuid from each rank and division and get 50 matches for that division. If that player does not have the 50 matches, then we will use next puuid from that rank and division list that we will generate from this notebook.
        2) 

# Importing the needed Libraries

In [1]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import time
from datetime import datetime

load_dotenv()
api_key = os.environ.get('riot_api_key')

# Function that saves the IDs to CSV files Locally

In [2]:
def save_players_by_division_to_csv(players_dict, base_path=None):
    """Save each division's players to separate CSV files."""
    
    if base_path is None:
        current_dir = os.getcwd()
        base_path = os.path.abspath(os.path.join(current_dir, '..', 'data', 'data collected', 'player ids'))
    
    os.makedirs(base_path, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    for division_key, players in players_dict.items():
        if not players:
            continue
            
        rows = []
        for p in players:
            rows.append({
                'tier': p.get('tier'),
                'rank': p.get('rank'),
                'puuid': p.get('puuid'),
                'queueType': p.get('queueType'),
                'region':p.get('region'),
                'sub_region':p.get('sub_region')
            })
        
        df = pd.DataFrame(rows)
        filename = f"{division_key}_{timestamp}.csv"
        filepath = os.path.join(base_path, filename)
        df.to_csv(filepath, index=False)
        print(f"Saved {len(df)} players to: {filepath}")


# Get 10 players at each rank + division from IRON_I to Diamond_IV

### Note that theses rank/tiers has 4 divisions

In [3]:
def get_x_players_per_low_tier_rank_and_division(region='europe', sub_region='eun1', queue_type='RANKED_SOLO_5x5', count=10):
    tiers = ['IRON', 'BRONZE', 'SILVER', 'GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND']
    divisions = ['I', 'II', 'III', 'IV']
    results = {}
    
    for tier in tiers:
        for division in divisions:
            key = f"{tier}_{division}"
            print(f"\n{tier} {division}:")
            players = []
            
            while len(players) < count:
                root_url = f"https://{sub_region}.api.riotgames.com/" 
                end_point = f"lol/league/v4/entries/{queue_type}/{tier}/{division}"
                query_params = f"?page=1&api_key={api_key}"
                response = requests.get(root_url + end_point + query_params)
                
                if response.status_code == 429:
                    time.sleep(int(response.headers.get('Retry-After', 10)))
                    continue
                if response.status_code != 200 or not response.json():
                    break
                
                active = [p for p in response.json() if not p.get('inactive', True)]
                for p in active:
                    p['region'] = region
                    p['sub_region'] = sub_region
                needed = count - len(players)
                players.extend(active[:needed])
                print(f"{min(len(active), needed)} players (total: {len(players)})")
                time.sleep(1.2)
            
            results[key] = players[:count]
    
    return results

low_tier_players = get_x_players_per_low_tier_rank_and_division(count=10)
low_tier_players


IRON I:
10 players (total: 10)

IRON II:
10 players (total: 10)

IRON III:
10 players (total: 10)

IRON IV:
10 players (total: 10)

BRONZE I:
10 players (total: 10)

BRONZE II:
10 players (total: 10)

BRONZE III:
10 players (total: 10)

BRONZE IV:
10 players (total: 10)

SILVER I:
10 players (total: 10)

SILVER II:
10 players (total: 10)

SILVER III:
10 players (total: 10)

SILVER IV:
10 players (total: 10)

GOLD I:
10 players (total: 10)

GOLD II:
10 players (total: 10)

GOLD III:
10 players (total: 10)

GOLD IV:
10 players (total: 10)

PLATINUM I:
10 players (total: 10)

PLATINUM II:
10 players (total: 10)

PLATINUM III:
10 players (total: 10)

PLATINUM IV:
10 players (total: 10)

EMERALD I:
10 players (total: 10)

EMERALD II:
10 players (total: 10)

EMERALD III:
10 players (total: 10)

EMERALD IV:
10 players (total: 10)

DIAMOND I:
10 players (total: 10)

DIAMOND II:
10 players (total: 10)

DIAMOND III:
10 players (total: 10)

DIAMOND IV:
10 players (total: 10)


{'IRON_I': [{'queueType': 'RANKED_SOLO_5x5',
   'tier': 'IRON',
   'rank': 'I',
   'puuid': 'z9ujdJ-YyyJq9oZVWikBeYvsQjHFjllJJ_9FEFQoBGUdejHcmpgOhdpEZmGyIrdQ9o78tGTW1Qs3bA',
   'leaguePoints': 78,
   'wins': 5,
   'losses': 9,
   'veteran': False,
   'inactive': False,
   'freshBlood': False,
   'hotStreak': False,
   'region': 'europe',
   'sub_region': 'eun1'},
  {'queueType': 'RANKED_SOLO_5x5',
   'tier': 'IRON',
   'rank': 'I',
   'puuid': 'hOJPjKWrmxs_zIPrVhMZjJvtAGMfhxhS69AdKtf49Q7_6cHvIPkjXQMCa_SlqJItDKPsfDeld_vP_w',
   'leaguePoints': 50,
   'wins': 1,
   'losses': 5,
   'veteran': False,
   'inactive': False,
   'freshBlood': False,
   'hotStreak': False,
   'region': 'europe',
   'sub_region': 'eun1'},
  {'queueType': 'RANKED_SOLO_5x5',
   'tier': 'IRON',
   'rank': 'I',
   'puuid': 'gk9npKHOTcK8Ck_oQFhb993o7ZUjjfgLKquAuVNG9BoD_Zfi6A4VoeaV892jaUNKN7HI1XCWoaQjpg',
   'leaguePoints': 25,
   'wins': 20,
   'losses': 50,
   'veteran': False,
   'inactive': False,
   'freshBlood':

In [4]:
save_players_by_division_to_csv(low_tier_players)

Saved 10 players to: c:\Users\Ibrahim Hegazi\Desktop\LOL Data Analytics\data\data collected\player ids\IRON_I_20260613_160057.csv
Saved 10 players to: c:\Users\Ibrahim Hegazi\Desktop\LOL Data Analytics\data\data collected\player ids\IRON_II_20260613_160057.csv
Saved 10 players to: c:\Users\Ibrahim Hegazi\Desktop\LOL Data Analytics\data\data collected\player ids\IRON_III_20260613_160057.csv
Saved 10 players to: c:\Users\Ibrahim Hegazi\Desktop\LOL Data Analytics\data\data collected\player ids\IRON_IV_20260613_160057.csv
Saved 10 players to: c:\Users\Ibrahim Hegazi\Desktop\LOL Data Analytics\data\data collected\player ids\BRONZE_I_20260613_160057.csv
Saved 10 players to: c:\Users\Ibrahim Hegazi\Desktop\LOL Data Analytics\data\data collected\player ids\BRONZE_II_20260613_160057.csv
Saved 10 players to: c:\Users\Ibrahim Hegazi\Desktop\LOL Data Analytics\data\data collected\player ids\BRONZE_III_20260613_160057.csv
Saved 10 players to: c:\Users\Ibrahim Hegazi\Desktop\LOL Data Analytics\data\

# Get 10 players at each rank + division from Master to Challenger

### Note that theses rank/tiers has 1 division

In [5]:
def get_x_puuids_per_high_tier(region='europe', sub_region='eun1', queue_type='RANKED_SOLO_5x5', count=10):
    """Get count PUUIDs from each high tier (Master, Grandmaster, Challenger)."""
    
    root = f'https://{sub_region}.api.riotgames.com/'
    tiers = {
        'CHALLENGER': 'lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5',
        'GRANDMASTER': 'lol/league/v4/grandmasterleagues/by-queue/RANKED_SOLO_5x5',
        'MASTER': 'lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5'
    }
    
    results = {}
    
    for tier_name, endpoint in tiers.items():
        key = f"{tier_name}_I"
        print(f"\n{tier_name}:")
        
        resp = requests.get(root + endpoint + '?api_key=' + api_key)
        
        if resp.status_code != 200:
            print(f"  Error: {resp.status_code}")
            results[key] = []
            continue
        
        data = resp.json()
        queue_type = data['queue']
        tier = data['tier']
        
        entries = data['entries']
        sorted_entries = sorted(entries, key=lambda x: x['leaguePoints'], reverse=True)
        
        players = []
        for p in sorted_entries[:count]:
            players.append({
                'puuid': p['puuid'],
                'tier': tier,
                'rank': p['rank'],
                'queueType': queue_type,
                'region': region,
                'sub_region': sub_region
            })
        
        print(f"  Got {len(players)} players")
        results[key] = players
    
    return results

high_tier_players = get_x_puuids_per_high_tier()
high_tier_players


CHALLENGER:
  Got 10 players

GRANDMASTER:
  Got 10 players

MASTER:
  Got 10 players


{'CHALLENGER_I': [{'puuid': 'GbaCpzcdYf_o_wq8mKclvvvWWO2P8DamlJbOSbWeLHIKIzJMNzxYdnG7tGk22K_JQ0YTXxBaMWUbzQ',
   'tier': 'CHALLENGER',
   'rank': 'I',
   'queueType': 'RANKED_SOLO_5x5',
   'region': 'europe',
   'sub_region': 'eun1'},
  {'puuid': 'dlXJYnkOwPwf_E4ZPCJoKOCNf9OjrmRsUl4isZq4ZEDBsB5lM3gOBRNoidn_G2VSaJXEYUtuCwsG8A',
   'tier': 'CHALLENGER',
   'rank': 'I',
   'queueType': 'RANKED_SOLO_5x5',
   'region': 'europe',
   'sub_region': 'eun1'},
  {'puuid': 'xg5UY3GLwcO-Ccl61Rfl_r6m7T3wNQcAuWYk7hEAJ3R_GrLakaitJEZaSpt4pLvZ8Wvc44h4CkCGHw',
   'tier': 'CHALLENGER',
   'rank': 'I',
   'queueType': 'RANKED_SOLO_5x5',
   'region': 'europe',
   'sub_region': 'eun1'},
  {'puuid': 'Z42-86-sjAP2sNK7cdecCcFiK7zojYS6gG6fd6dMNf5BP6-CL4hgTWBeS7plAprHaDAmnb6IHcpmVw',
   'tier': 'CHALLENGER',
   'rank': 'I',
   'queueType': 'RANKED_SOLO_5x5',
   'region': 'europe',
   'sub_region': 'eun1'},
  {'puuid': 'QE8sRyQW044Zait6UA914zZDF802LfitL6bOlFuFJdyF9jf6cYgBWNsUXTz0SYI9JAW6Tlp5eDIE3Q',
   'tier': 'CH

In [6]:
save_players_by_division_to_csv(high_tier_players)

Saved 10 players to: c:\Users\Ibrahim Hegazi\Desktop\LOL Data Analytics\data\data collected\player ids\CHALLENGER_I_20260613_160109.csv
Saved 10 players to: c:\Users\Ibrahim Hegazi\Desktop\LOL Data Analytics\data\data collected\player ids\GRANDMASTER_I_20260613_160109.csv
Saved 10 players to: c:\Users\Ibrahim Hegazi\Desktop\LOL Data Analytics\data\data collected\player ids\MASTER_I_20260613_160109.csv


# Merging/Combining all the CSV files together 

In [7]:
def merge_division_files(base_path=None):
    """Merge all division CSV files into a single CSV maintaining rank order."""
    
    if base_path is None:
        current_dir = os.getcwd()
        base_path = os.path.abspath(os.path.join(current_dir, '..', 'data', 'data collected', 'player ids'))
    
    # Define order: Iron to Challenger
    tier_order = ['IRON', 'BRONZE', 'SILVER', 'GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND', 'MASTER', 'GRANDMASTER', 'CHALLENGER']
    div_order = ['I', 'II', 'III', 'IV']
    
    # Build ordered division list
    ordered_divisions = []
    for tier in tier_order:
        if tier in ['MASTER', 'GRANDMASTER', 'CHALLENGER']:
            ordered_divisions.append(f"{tier}_I")
        else:
            for div in div_order:
                ordered_divisions.append(f"{tier}_{div}")
    
    # Find latest file for each division
    all_files = os.listdir(base_path)
    division_files = {}
    
    for div in ordered_divisions:
        matching = [f for f in all_files if f.startswith(div) and f.endswith('.csv')]
        if matching:
            # Get most recent file (last timestamp)
            matching.sort(reverse=True)
            division_files[div] = os.path.join(base_path, matching[0])
    
    # Read and merge
    dfs = []
    for div in ordered_divisions:
        if div in division_files:
            df = pd.read_csv(division_files[div])
            df['division'] = div
            dfs.append(df)
            print(f"Loaded {div}: {len(df)} players")
    
    merged = pd.concat(dfs, ignore_index=True)
    
    # Save merged file
    from datetime import datetime
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_path = os.path.join(base_path, f"Merged_Player_IDs_{timestamp}.csv")
    merged.to_csv(output_path, index=False)
    
    print(f"\nMerged {len(merged)} players to: {output_path}")
    return merged

# Usage
all_players = merge_division_files()

Loaded IRON_I: 10 players
Loaded IRON_II: 10 players
Loaded IRON_III: 10 players
Loaded IRON_IV: 10 players
Loaded BRONZE_I: 10 players
Loaded BRONZE_II: 10 players
Loaded BRONZE_III: 10 players
Loaded BRONZE_IV: 10 players
Loaded SILVER_I: 10 players
Loaded SILVER_II: 10 players
Loaded SILVER_III: 10 players
Loaded SILVER_IV: 10 players
Loaded GOLD_I: 10 players
Loaded GOLD_II: 10 players
Loaded GOLD_III: 10 players
Loaded GOLD_IV: 10 players
Loaded PLATINUM_I: 10 players
Loaded PLATINUM_II: 10 players
Loaded PLATINUM_III: 10 players
Loaded PLATINUM_IV: 10 players
Loaded EMERALD_I: 10 players
Loaded EMERALD_II: 10 players
Loaded EMERALD_III: 10 players
Loaded EMERALD_IV: 10 players
Loaded DIAMOND_I: 10 players
Loaded DIAMOND_II: 10 players
Loaded DIAMOND_III: 10 players
Loaded DIAMOND_IV: 10 players
Loaded MASTER_I: 10 players
Loaded GRANDMASTER_I: 10 players
Loaded CHALLENGER_I: 10 players

Merged 310 players to: c:\Users\Ibrahim Hegazi\Desktop\LOL Data Analytics\data\data collected\